# 19 - Sensitivity analysis 3: competing event (PK vs PD mechanism)

As notebook 17, by mechanism of interaction.

**Manuscript items:** Fig S11c

**Inputs:** derived cohort files from 01, comorbidity table from 02, CPRD DrugIssue extract, product dictionary, dosage lookup, interacting-drug code lists

**Outputs:** `SENS_COMPETING_DIR/hr_drug_pk_pd_minor_summary.xlsx`

*Derived from the original analysis notebook `sensitivity_analysis/competing_risk/hr_pk_pd.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
# Cap BLAS/OpenMP threads BEFORE numpy is imported. Without this each worker
# process below spawns its own thread pool and the machine thrashes.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter


In [ ]:
major_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), dtype=str)
major_doac_exposure["age_at_study_start"] = major_doac_exposure["age_at_study_start"].astype(int)
major_doac_exposure["study_start_date"] = pd.to_datetime(major_doac_exposure["study_start_date"])
major_doac_exposure["study_end_date"] = pd.to_datetime(major_doac_exposure["study_end_date"])
major_doac_exposure["event_date"] = pd.to_datetime(major_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

major_doac_exposure = major_doac_exposure[(major_doac_exposure["age_at_study_start"] >= 18) &
                (major_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (major_doac_exposure["taken_warfarin_1w_before"] =="0") &
                 (major_doac_exposure["taken_warfarin_during_study"] =="0") ]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
major_doac_exposure = major_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]
print(major_doac_exposure.shape[0])

In [ ]:
major_events_df = major_doac_exposure[["patid", "event_date"]].copy()
major_events_df.dropna(inplace=True)
major_events_df.reset_index(drop=True, inplace=True)
major_events_df

In [ ]:
minor_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx"), dtype=str)
minor_doac_exposure["age_at_study_start"] = minor_doac_exposure["age_at_study_start"].astype(int)
minor_doac_exposure["study_start_date"] = pd.to_datetime(minor_doac_exposure["study_start_date"])
minor_doac_exposure["study_end_date"] = pd.to_datetime(minor_doac_exposure["study_end_date"])
minor_doac_exposure["event_date"] = pd.to_datetime(minor_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

minor_doac_exposure = minor_doac_exposure[(minor_doac_exposure["age_at_study_start"] >= 18) &
                (minor_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (minor_doac_exposure["taken_warfarin_1w_before"] =="0") &
                    (minor_doac_exposure["taken_warfarin_during_study"] =="0")]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
minor_doac_exposure = minor_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["doac_start_date"] = pats_data["patid"].map(major_doac_exposure.set_index("patid")["study_start_date"])
pats_data["doac_start_date"] = pd.to_datetime(pats_data["doac_start_date"], errors="coerce")
pats_data = pats_data[pats_data["patid"].isin(major_doac_exposure["patid"])]
pats_data["age"] = (pats_data["doac_start_date"] - pd.to_datetime(pats_data["yob"], format="%Y")).dt.days // 365.25
## add the comorbidities to major_doac_exposure

In [ ]:
a = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_DrugInteractions_Table.xlsx"), dtype=str)
a = a[["Drug", "Category", "Action", "PK/PD"]]

drugs_action_increase = a[a["Action"] == "Increased"]
drugs_action_decrease = a[a["Action"] == "Decreased"]
p_k = drugs_action_increase[drugs_action_increase["PK/PD"] == "PK"]
p_d = drugs_action_increase[drugs_action_increase["PK/PD"] == "PD"]

In [ ]:
interacting_drugs = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_interacting_drugs.xlsx"), dtype=str)
interacting_drugs = interacting_drugs[interacting_drugs["DrugSubstanceName"].str.contains("/", case=False, na=False) == False]
interacting_drugs["DrugSubstanceName"] = np.where(interacting_drugs["DrugSubstanceName"].isna(), interacting_drugs["matched_drugs"], interacting_drugs["DrugSubstanceName"])

interacting_drugs = interacting_drugs[
    interacting_drugs.apply(
        lambda row: row["DrugSubstanceName"].startswith(row["matched_drugs"][:3])
        if pd.notna(row["DrugSubstanceName"]) else False,
        axis=1
    )
]
words_to_remove = ["eye drops", "inhaler", "cream", "ointment", "gel", "spray", "lotion", "shampoo", "cream"]
interacting_drugs = interacting_drugs[interacting_drugs["Term from EMIS"].str.contains("|".join(words_to_remove), case=False, na=False) == False]
interacting_drugs_codes = interacting_drugs[["ProdCodeId", "matched_drugs"]]
interacting_drugs_codes.rename(columns={"matched_drugs": "drug_name", "ProdCodeId":"prodcodeid"}, inplace=True)
interacting_drugs_codes.drop_duplicates(subset=["prodcodeid"], inplace=True)
interacting_drugs_codes["drug_class"] = interacting_drugs_codes["drug_name"].map(a.set_index("Drug")["Category"])
interacting_drugs_codes

In [ ]:
def calculate_enddate(df, dosage):
    df = df.copy()
    df["daily_dose"] = df["dosageid"].map(dosage.set_index("dosageid")["daily_dose"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["daily_dose"] = pd.to_numeric(df["daily_dose"], errors="coerce").fillna(0)

    # duration==0 rows that CAN be resolved from quantity / daily dose
    mask1 = (
        (df["duration"] == 0)
        & df["quantity"].notna()
        & (df["daily_dose"] != 0)
        & (df["quantity"] != 0)
    )
    # All other duration==0 rows (daily_dose unknown/zero, OR quantity missing/zero)
    # are imputed from the per-product median below, rather than being silently
    # left at 0 (which would otherwise produce a same-day enddate for an
    # unknown/unresolvable duration).
    mask2 = (df["duration"] == 0) & (~mask1)

    df["new_duration"] = np.where(mask1, df["quantity"] / df["daily_dose"], df["duration"])
    df["new_duration"] = pd.to_numeric(df["new_duration"], errors="coerce")

    # Median computed only from rows with a genuinely recorded (non-zero) duration,
    # so neither mask1-resolved rows nor other zero-duration rows distort it.
    median_durations = (
        df.loc[df["duration"] != 0].groupby("prodcodeid")["duration"].median()
    )

    df["new_duration"] = np.where(mask2, df["prodcodeid"].map(median_durations), df["new_duration"])

    df["enddate"] = (pd.to_datetime(df["issuedate"]) + pd.to_timedelta(df["new_duration"], unit="D"))

    return df

In [ ]:
import zipfile
dosage = pd.read_csv(os.path.join(LOOKUP_DIR, "common_dosages.txt"), sep="\t",dtype=str)


zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(interacting_drugs_codes["prodcodeid"])) & (df['patid'].isin(minor_doac_exposure["patid"].tolist()))]

    df["study_start_date"] = df["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
    df["study_end_date"] = df["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
    
    df = calculate_enddate(df, dosage)

    df = df[(df["issuedate"] < df["study_end_date"]) &(df["enddate"] > df["study_start_date"])]


    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Combine results
    interacting_drugs_exposure = pd.concat(all_results, ignore_index=True)

interacting_drugs_exposure["enddate"] = pd.to_datetime(interacting_drugs_exposure["enddate"], errors='coerce')
interacting_drugs_exposure["enddate"] = interacting_drugs_exposure["enddate"] + pd.to_timedelta(1, unit='D')
interacting_drugs_exposure.drop(columns=["dosageid", "duration", "quantity", "daily_dose", "probobsid", "new_duration"], inplace=True)
interacting_drugs_exposure.sort_values(['patid', 'issuedate'], inplace=True)
interacting_drugs_exposure.reset_index(drop=True, inplace=True)
interacting_drugs_exposure["drug_class"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_class"])
interacting_drugs_exposure["drug_name"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_name"])
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_name"].isin(drugs_action_increase["Drug"].tolist())]

In [ ]:
def process_group(group):
    group = group.sort_values('issuedate').reset_index(drop=True)
    merged = []

    # Set days_gap based on drug_class
    drug_class = group['drug_class'].iloc[0].lower()
    if drug_class == 'antibiotic':
        days_gap = 7
    elif drug_class == 'antifungal':
        days_gap = 7
    else:
        days_gap = 14  # default

    current_start = group.loc[0, 'issuedate']
    current_end = group.loc[0, 'enddate']

    for i in range(1, len(group)):
        next_start = group.loc[i, 'issuedate']
        next_end = group.loc[i, 'enddate']

        # Merge intervals if gap < days_gap
        if (next_start - current_end).days < days_gap:
            current_end = max(current_end, next_end)
        else:
            merged.append({
                'patid': group.loc[i-1, 'patid'],
                'drug_name': group.loc[i-1, 'drug_name'],
                'pk_pd': group.loc[i-1, 'pk_pd'],
                'drug_class': group.loc[i-1, 'drug_class'],
                'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                'enddate': min(current_end, group['study_end_date'].iloc[0])
            })
            current_start = next_start
            current_end = next_end

    # Append last interval
    merged.append({
        'patid': group.loc[len(group)-1, 'patid'],
        'drug_name': group.loc[len(group)-1, 'drug_name'],
        'pk_pd': group.loc[len(group)-1, 'pk_pd'],
        'drug_class': group.loc[len(group)-1, 'drug_class'],
        'issuedate': max(current_start, group['study_start_date'].iloc[0]),
        'enddate': min(current_end, group['study_end_date'].iloc[0])
    })

    return pd.DataFrame(merged)


def merge_prescriptions(df):
    # Ensure dates are datetime
    for col in ['issuedate', 'enddate', 'study_start_date', 'study_end_date']:
        df[col] = pd.to_datetime(df[col])

    # ⭐ Group by patid + pk_pd instead of patid only
    groups = [group for _, group in df.groupby(['patid', 'pk_pd'])]

    # Multiprocessing
    with Pool(20) as pool:
        result_list = pool.map(process_group, groups)

    merged_df = pd.concat(result_list, ignore_index=True)
    return merged_df


In [ ]:
def _process_patient(args):
    patid, group = args
    
    doac_start = group["study_start_date"].iloc[0]
    doac_end = group["study_end_date"].iloc[0]

    # clip exposures to DOAC period
    group = group.copy()
    group["start"] = group["issuedate"].clip(lower=doac_start, upper=doac_end)
    group["stop"] = group["enddate"].clip(lower=doac_start, upper=doac_end)

    # collect cut points
    cut_points = sorted(set([doac_start, doac_end]) | set(group["start"]) | set(group["stop"]))
    intervals = [(cut_points[i], cut_points[i+1]) for i in range(len(cut_points)-1)]

    rows = []
    for start, stop in intervals:
        if start >= stop:
            continue
        active = group[(group["start"] <= start) & (group["stop"] >= stop)]
        active_mechanism = sorted(active["pk_pd"].unique().tolist())    # changed drug_name to pk_pd
        active_drugs = sorted(active["drug_name"].unique().tolist()) 
        rows.append({
            "patid": patid,
            "start_date": start,
            "end_date": stop,
            "concomitant_drugs_ls": active_drugs,
            "active_mechanism": active_mechanism
        })
    return rows

def build_exposure_timeline(df, n_jobs=None):
    df = df.copy()
    df['issuedate'] = pd.to_datetime(df['issuedate'])
    df['enddate'] = pd.to_datetime(df['enddate'])
    df['study_start_date'] = pd.to_datetime(df['study_start_date'])
    df['study_end_date'] = pd.to_datetime(df['study_end_date'])
    
    grouped = list(df.groupby('patid'))

    with mp.Pool(processes=20) as pool:
        results = pool.map(_process_patient, grouped)

    # flatten list of lists
    flat_results = [row for sublist in results for row in sublist]
    return pd.DataFrame(flat_results)

In [ ]:
# ===========================================================================
# Restore patients who never received an interacting drug
# ===========================================================================
# The Methods define the comparator as "DOAC treatment without any interacting
# drug". build_exposure_timeline() only emits rows for patients who had at
# least one interacting-drug prescription, so without this step the reference
# stratum is built solely from the drug-free gaps of EXPOSED patients.
#
# Those two populations are not interchangeable. Patients who are never
# prescribed an interacting drug have fewer comorbidities and less healthcare
# contact, so their bleeding rate is lower. Omitting them inflates the
# reference rate, which biases every hazard ratio towards the null and makes
# these estimates non-comparable with the drug-level analysis, where the
# never-exposed patients ARE included.

def add_never_exposed(timeline_df, cohort_df, exposure_zero_cols=(),
                      category_col=None, category_value="0_Drug", label=""):
    """Give never-exposed patients their whole DOAC follow-up as one
    unexposed interval, and append it to the timeline."""
    exposed = set(timeline_df["patid"].unique())
    never = cohort_df.drop_duplicates(subset=["patid"])
    never = never[~never["patid"].isin(exposed)].copy()

    if never.empty:
        print(f"[{label}] no never-exposed patients to add")
        return timeline_df

    never["start_date"] = never["study_start_date"]
    never["end_date"] = never["study_end_date"]
    never["event_flag"] = np.where(
        (never["event_date"] >= never["start_date"]) &
        (never["event_date"] <= never["end_date"]), 1, 0)

    if "concomitant_drugs_ls" in timeline_df.columns:
        never["concomitant_drugs_ls"] = [[] for _ in range(len(never))]
    for c in exposure_zero_cols:
        never[c] = 0
    if category_col is not None:
        never[category_col] = category_value

    out = pd.concat([timeline_df, never], ignore_index=True)
    for c in exposure_zero_cols:                       # bool -> int after concat
        out[c] = out[c].fillna(0).astype(int)

    py = ((never["end_date"] - never["start_date"]).dt.total_seconds()
          / 86400 / 365.25).sum()
    print(f"[{label}] added {len(never):,} never-exposed patients "
          f"({py:,.0f} person-years, {int(never['event_flag'].sum()):,} events). "
          f"Timeline now covers {out['patid'].nunique():,} patients.")
    return out.reset_index(drop=True)


In [ ]:
# ===========================================================================
# Baseline covariates - RAW values, missing continuous values left as NaN
# ===========================================================================
# This cell no longer reads a pre-imputed workbook. It loads the raw comorbidity
# table so the missing values are still present and can be handled by the
# multiple imputation further down. Imputation has to happen after the analysis
# frames exist, because the imputation model needs the outcome and the
# Nelson-Aalen cumulative hazard (White & Royston 2009).

comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype={"patid":str})

binary_comorbidity_cols = ["major_bled_hist", "hypertension", "liver_disease", "stroke", "gender", "heart_failure",
                           "atrial_fibrillation", "venous_thromboembolism", "cancer", "diabetes", "smoking", "alcohol"]
continuous_comorbidity_cols = ["age_at_study_start", "ckd_value", "bmi_value"]

comorbidities["gender"] = comorbidities["patid"].map(pats_data.set_index("patid")["gender"])

# Blanks / sentinel strings become NaN rather than surviving as object dtype.
for col in continuous_comorbidity_cols:
    if col in comorbidities.columns:
        comorbidities[col] = pd.to_numeric(comorbidities[col], errors="coerce")


def backfill_age(df, cohort_df, label=""):
    """Age is derivable exactly from year of birth and the index date, so it is
    merged in rather than imputed. Imputing a variable you can compute would not
    survive review."""
    before = int(df["age_at_study_start"].isna().sum()) if "age_at_study_start" in df else len(df)
    age = pd.to_numeric(df.get("age_at_study_start"), errors="coerce")

    if "age_at_study_start" in cohort_df.columns:
        lut = cohort_df.drop_duplicates("patid").set_index("patid")["age_at_study_start"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut, errors="coerce")))

    # Second fallback: recompute from yob in pats_data (column is named "age" here).
    if age.isna().any() and "age" in pats_data.columns:
        lut2 = pats_data.drop_duplicates("patid").set_index("patid")["age"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut2, errors="coerce")))

    df["age_at_study_start"] = age
    after = int(age.isna().sum())
    print(f"[{label}] age missing rows: {before:,} -> {after:,}"
          + ("  <-- still unresolved, investigate before imputing" if after else ""))
    return df


def report_missingness(df, label=""):
    """Patient-level missingness. Cross-check against the manuscript (eGFR 3.7%,
    BMI 8.4%): a mismatch points at the merge, not at the data."""
    pat = df.drop_duplicates("patid")
    rep = pd.DataFrame([
        {"variable": c, "n_missing": int(pat[c].isna().sum()),
          "pct_missing": round(100 * pat[c].isna().mean(), 2)}
        for c in continuous_comorbidity_cols])
    print(f"[{label}] {pat['patid'].nunique():,} patients")
    print(rep.to_string(index=False))
    return rep

# ---------------------------------------------------------------------------
# Covariates are attached to the analysis frame in the MINOR BLEEDING section
# below. The serious and all-bleeding frames are no longer built in this
# notebook, so the block that mapped these columns onto major_df / all_df has
# been removed; only the definitions above (comorbidities, the column lists,
# backfill_age, report_missingness) are needed downstream.
# ---------------------------------------------------------------------------


In [ ]:
drugs_covariates = " + ".join(["p_k_drug", "p_d_drug"])
comorbidities_covariates = " + ".join(binary_comorbidity_cols + continuous_comorbidity_cols)

covariates = drugs_covariates + " + " + comorbidities_covariates

drug_classes_to_consider = ["p_k_drug", "p_d_drug"]

# Columns the adjusted fit needs, so workers slice rather than copy everything.
adjusted_cols = (["patid", "start_date", "end_date", "event_flag"]
                 + drug_classes_to_consider
                 + binary_comorbidity_cols + continuous_comorbidity_cols)


In [ ]:
# ---------- Helper functions (defined once, before any use) ----------
from scipy import stats

def calculate_crude_hr(df, drug_class):
    ctv_crude = CoxTimeVaryingFitter()
    ctv_crude.fit(
        df,
        id_col='patid',
        start_col='start_date',
        stop_col='end_date',
        event_col='event_flag',
        formula=drug_class,
        show_progress=True
    )
    row = ctv_crude.summary.loc[drug_class]
    return row["exp(coef)"], row["exp(coef) lower 95%"], row["exp(coef) upper 95%"], row["p"]


def _as_list(x):
    """concomitant_drugs_ls holds python lists, but NaN can survive a concat."""
    return x if isinstance(x, (list, tuple, set)) else []


def reference_mask(df, all_terms=None):
    """Boolean mask over interval rows: person-time with NO interacting drug active.

    This is the reference level the mutually adjusted Cox models actually
    contrast against - every interacting-drug indicator is 0 - and so it is the
    only comparator that makes an absolute measure consistent with the hazard
    ratio sitting next to it in the table.

    `df[drug] == 0` is NOT this comparator. It also contains person-time exposed
    to the *other* interacting drugs, which carries a higher event rate. Using it
    is what made the old comparator rate come out ABOVE the cohort-wide rate
    (~1.87 vs 1.8 per 100 PY), which is arithmetically impossible for a genuine
    unexposed stratum when every exposed stratum sits above the cohort mean.

    Derived from `concomitant_drugs_ls` where available so that the mask does not
    depend on which subset of terms is being tabulated (build_split_ppi_summary
    tabulates the NSAID/aspirin rows separately from the rest, and both must
    share one reference).
    """
    # Prefer the model's own exposure terms: all terms at zero IS the reference
    # level of the fitted model, by definition. concomitant_drugs_ls is only a
    # fallback - in the mechanism and drug-count notebooks the model terms are
    # derived quantities (p_k_drug/p_d_drug, 1_Drug/2_Drugs/3_or_more_Drugs) and
    # the raw drug list would not give the same stratum.
    if all_terms:
        cols = [t for t in all_terms if t in df.columns]
        if not cols:
            raise ValueError("None of `all_terms` are columns of df.")
        return (df[cols].sum(axis=1) == 0).to_numpy(dtype=bool)
    if "concomitant_drugs_ls" in df.columns:
        return df["concomitant_drugs_ls"].map(
            lambda v: len(_as_list(v)) == 0).to_numpy(dtype=bool)
    raise ValueError(
        "Cannot identify unexposed person-time: pass `all_terms`, or keep the "
        "`concomitant_drugs_ls` column on the analysis frame.")


def _events_and_py(df, mask):
    """Events (unique patients) and person-years over a set of interval rows."""
    sub = df.loc[mask]
    if len(sub) == 0:
        return 0, 0.0, np.nan
    # total_seconds rather than .dt.days: .days truncates every interval towards
    # zero, which silently shortens person-time across millions of rows.
    days = ((sub["end_date"] - sub["start_date"]).dt.total_seconds() / 86400.0).clip(lower=0)
    py = float(days.sum()) / 365.25
    events = int(sub.loc[sub["event_flag"] == 1, "patid"].nunique())
    rate100 = (events / py) * 100 if py > 0 else np.nan
    return events, py, rate100


def calculate_events_person_years(df, drug_class, exposed=True):
    """Crude events / person-years / rate per 100 PY for one term.

    Kept for the descriptive 'Events/Person Years' and 'Events per 100 PY'
    columns. `exposed=False` means 'not on this drug' and is NOT the model
    reference - use reference_mask() for anything compared against a hazard
    ratio.
    """
    flag = 1 if exposed else 0
    return _events_and_py(df, (df[drug_class] == flag).to_numpy(dtype=bool))


def _se_and_crit_from_row(row):
    """Standard error of log HR and the two-sided 95% critical value.

    Uses the model's own SE. After Rubin pooling the interval is built on a
    Barnard-Rubin t quantile rather than 1.96, so the same quantile is reused
    here; otherwise the rate-difference CI would not be on the same footing as
    the HR CI printed beside it.
    """
    se = float(row["se(coef)"]) if "se(coef)" in row.index and pd.notna(row["se(coef)"]) else np.nan
    if not np.isfinite(se):
        lo, hi = float(row["exp(coef) lower 95%"]), float(row["exp(coef) upper 95%"])
        se = (np.log(hi) - np.log(lo)) / (2 * 1.959963985)
    dof = float(row["df"]) if "df" in row.index and pd.notna(row["df"]) else np.inf
    crit = float(stats.t.ppf(0.975, dof)) if np.isfinite(dof) else 1.959963985
    return se, crit


def calculate_nnh(df, drug_class, model_row, all_terms=None, reference=None, scale=1000):
    """Adjusted rate difference and person-years-per-additional-event.

    The absolute measure is now derived from the adjusted hazard ratio rather
    than from two crude rates:

        RD = r0 * (HR_adj - 1)

    where r0 is the event rate in unexposed person-time (no interacting drug
    active) and HR_adj is the mutually adjusted, MI-pooled hazard ratio for this
    drug. Under the proportional hazards model the adjusted rate in exposed time
    is r0 * HR_adj, so RD is the excess rate attributable to the exposure with
    the confounders held fixed. The previous version subtracted two crude rates
    and never touched the model at all, so the number labelled "adjusted rate
    difference" in the manuscript was not adjusted for anything.

    CI by the delta method on RD = exp(a) * (exp(b) - 1), a = log r0, b = log HR:

        Var(RD) = r0^2 * [ (HR - 1)^2 * Var(log r0) + HR^2 * Var(log HR) ]

    with Var(log r0) = 1 / events_ref (Poisson). r0 and HR_adj are treated as
    independent; they are weakly correlated in truth (the reference person-time
    contributes to the partial likelihood), but the correlation is negligible
    relative to the two variance terms and ignoring it is conservative here.

    NNH is returned in PERSON-YEARS of exposure per additional event. It is not a
    number of persons - report it under that name in the tables and text.
    """
    ref = reference_mask(df, all_terms) if reference is None else reference

    events_ref, py_ref, rate_ref_100 = _events_and_py(df, ref)
    events_exp, py_exp, rate_exp_100 = calculate_events_person_years(df, drug_class, exposed=True)

    hr = float(model_row["exp(coef)"])
    se_log_hr, crit = _se_and_crit_from_row(model_row)

    out = {
        "ref_events": events_ref,
        "ref_person_years": round(py_ref, 2),
        "rate_unexposed_per_100py": round(rate_ref_100, 3) if np.isfinite(rate_ref_100) else np.nan,
        "HR_used_for_RD": round(hr, 4),
    }

    if events_ref == 0 or py_ref <= 0 or not np.isfinite(hr):
        nan_cols = {
            f"rate_diff_per_{scale}py": np.nan,
            f"rate_diff_per_{scale}py_lower": np.nan,
            f"rate_diff_per_{scale}py_upper": np.nan,
            "NNH": "Not estimable", "NNH_CI_lower": np.nan, "NNH_CI_upper": np.nan,
            "CI_crosses_null": True,
        }
        return {**out, **nan_cols}

    r0 = rate_ref_100 / 100.0                      # events per person-year
    rd = r0 * (hr - 1.0)                           # adjusted rate difference

    var_log_r0 = 1.0 / events_ref
    var_rd = (r0 ** 2) * ((hr - 1.0) ** 2 * var_log_r0 + (hr ** 2) * se_log_hr ** 2)
    se_rd = np.sqrt(var_rd)
    rd_lower, rd_upper = rd - crit * se_rd, rd + crit * se_rd

    rd_s, rd_lo_s, rd_hi_s = rd * scale, rd_lower * scale, rd_upper * scale

    def _nnh(x):
        # x is a rate difference on the `scale` person-year basis; scale / x is
        # therefore person-years per event regardless of the scale chosen.
        return scale / x if np.isfinite(x) and x > 0 else np.nan

    nnh = _nnh(rd_s)
    nnh_lower = _nnh(rd_hi_s)      # bounds flip under the reciprocal
    nnh_upper = _nnh(rd_lo_s)
    crosses_null = (not np.isfinite(rd_lo_s)) or (rd_lo_s <= 0 <= rd_hi_s)

    out.update({
        f"rate_diff_per_{scale}py": round(rd_s, 3),
        f"rate_diff_per_{scale}py_lower": round(rd_lo_s, 3),
        f"rate_diff_per_{scale}py_upper": round(rd_hi_s, 3),
        # "Not estimable" when HR <= 1: no excess rate, so the reciprocal is
        # undefined. NaN in the upper bound means the RD interval crossed zero
        # and the bound runs to infinity.
        "NNH": round(nnh, 1) if np.isfinite(nnh) else "Not estimable",
        "NNH_CI_lower": round(nnh_lower, 1) if np.isfinite(nnh_lower) else np.nan,
        "NNH_CI_upper": round(nnh_upper, 1) if np.isfinite(nnh_upper) else np.nan,
        "CI_crosses_null": crosses_null,
    })
    return out


def calculate_crude_rate_difference(df, drug_class, scale=1000):
    """The old, unadjusted contrast, kept only so the two can be reported
    side by side. Comparator here is 'not on this drug', which still includes
    time on other interacting drugs - it is descriptive, not a causal contrast.
    """
    _, _, rate_exp_100 = calculate_events_person_years(df, drug_class, exposed=True)
    _, _, rate_unexp_100 = calculate_events_person_years(df, drug_class, exposed=False)
    rd = (rate_exp_100 - rate_unexp_100) / 100.0
    return {f"crude_rate_diff_per_{scale}py": round(rd * scale, 3),
            "crude_NNH": round(1.0 / rd, 1) if rd > 0 else "Not estimable"}


def build_bleeding_summary(ctv_model, df, drug_classes, nnh_scale=1000,
                           all_terms=None, include_crude_rd=True):
    # Accepts either a fitted CoxTimeVaryingFitter or a `.summary` DataFrame
    # (the parallel fitters return the latter, since a whole fitted model is
    # expensive to send back from a worker process).
    _raw = ctv_model if isinstance(ctv_model, pd.DataFrame) else ctv_model.summary
    summary = _raw.reset_index().rename(columns={
        "covariate": "drug_class", "exp(coef)": "HR",
        "exp(coef) lower 95%": "CI Lower", "exp(coef) upper 95%": "CI Upper",
    })
    summary = summary[summary["drug_class"].isin(drug_classes)].copy()

    summary["p"] = summary["p"].round(4)
    for col in ["HR", "CI Lower", "CI Upper"]:
        summary[col] = summary[col].round(2)
    summary = summary.sort_values("HR", ascending=False).reset_index(drop=True)

    # One reference stratum for the whole table, computed once. Every hazard
    # ratio in the model is a contrast against this same drug-free person-time,
    # so every absolute measure must use it too - including the rows that
    # build_split_ppi_summary tabulates from the PPI-adjusted fit.
    ref = reference_mask(df, all_terms)
    ref_events, ref_py, ref_rate = _events_and_py(df, ref)
    print(f"    reference (no interacting drug active): {ref_events:,} events / "
          f"{ref_py:,.0f} person-years = {ref_rate:.3f} per 100 PY")

    person_time_rows = []
    for drug in drug_classes:
        events, py, rate = calculate_events_person_years(df, drug, exposed=True)
        nnh_res = calculate_nnh(df, drug, _raw.loc[drug], reference=ref, scale=nnh_scale)
        row = {
            "drug_class": drug,
            "no_of_events": events,
            "person_years": round(py, 2),
            "events_per_100py": round(rate, 2),
            **nnh_res,
        }
        if include_crude_rd:
            row.update(calculate_crude_rate_difference(df, drug, scale=nnh_scale))
        person_time_rows.append(row)
    person_time_df = pd.DataFrame(person_time_rows)

    summary = summary.merge(person_time_df, on="drug_class", how="left")
    summary["person_years"] = summary["person_years"].fillna(0).astype(int)
    summary["events/person_years"] = summary["no_of_events"].astype(str) + "/" + summary["person_years"].astype(str)
    summary["Patient Counts"] = summary["drug_class"].map(lambda d: df[df[d] == 1]["patid"].nunique())
    summary["No of Events"] = summary["no_of_events"]

    nnh_cols = [c for c in person_time_df.columns
                if c not in ("drug_class", "no_of_events", "person_years", "events_per_100py")]

    return summary[[
        "drug_class", "Patient Counts", "events/person_years", "events_per_100py",
        "HR", "CI Lower", "CI Upper", "p", "No of Events",
        *nnh_cols,
    ]]


# ---------- Parallel Cox fitting ----------------------------------------------
# There is only ONE adjusted model per outcome here, so unlike the drug-name and
# drug-class notebooks there is no second adjusted fit to run alongside it.
# Instead the adjusted fit and all of the crude fits go into a single pool, so
# total time is roughly max(adjusted, crude) rather than adjusted + sum(crude).
# The adjusted fit dominates, so the crude fits become close to free.
#
# The data frame is published to a module-level global BEFORE the Pool is
# created: on Linux the workers are forked and inherit it copy-on-write, so a
# large frame is never pickled or duplicated. Same pattern the DrugIssue readers
# above already use. Requires the default 'fork' start method
# (mp.get_start_method() should return 'fork').

_INDEX_COLS = ["patid", "start_date", "end_date", "event_flag"]

_FIT_DF = None   # inherited by forked workers, never pickled


def _fit_worker(task):
    kind, name, formula, cols = task
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        _FIT_DF[cols],
        id_col='patid', start_col='start_date', stop_col='end_date',
        event_col='event_flag', formula=formula, show_progress=False,
    )
    if kind == "adjusted":
        return kind, name, ctv.summary
    row = ctv.summary.loc[name]
    return kind, name, {
        "drug_class": name,
        "Crude HR": round(row["exp(coef)"], 2),
        "Crude CI Lower": round(row["exp(coef) lower 95%"], 2),
        "Crude CI Upper": round(row["exp(coef) upper 95%"], 2),
        "Crude p": round(row["p"], 4),
    }


def fit_outcome(df, formula, cols, crude_terms, label="", n_jobs=None):
    """Fit the adjusted model and every crude model for one outcome, in parallel.

    Returns {"adjusted": summary DataFrame, "crude": DataFrame of crude HRs}.
    The adjusted entry is a `.summary` frame rather than a fitted model, because
    shipping a whole fitted model back from a worker is expensive;
    build_bleeding_summary accepts either.
    """
    global _FIT_DF
    crude_terms = list(crude_terms)
    tasks = [("adjusted", "adjusted", formula, cols)]
    tasks += [("crude", t, t, _INDEX_COLS + [t]) for t in crude_terms]

    t0 = time.time()
    _FIT_DF = df
    try:
        with Pool(min(n_jobs or len(tasks), len(tasks))) as pool:
            results = pool.map(_fit_worker, tasks)
    finally:
        _FIT_DF = None

    adjusted = next(r for k, _, r in results if k == "adjusted")
    crude_rows = {n: r for k, n, r in results if k == "crude"}
    crude_df = pd.DataFrame([crude_rows[t] for t in crude_terms])
    print(f"[{label}] 1 adjusted + {len(crude_terms)} crude models "
          f"in {time.time()-t0:.1f}s")
    return {"adjusted": adjusted, "crude": crude_df}


## Multiple imputation of missing continuous covariates

Missing values in the continuous baseline covariates (`ckd_value` / eGFR and
`bmi_value`) are handled by **multiple imputation by chained equations (MICE)**
using `miceforest` (random forest / LightGBM chained equations with predictive
mean matching), rather than by complete-case analysis or a missing-indicator
category.

Three things make this survival-analysis-appropriate rather than a generic MICE run:

1. **The outcome is in the imputation model.** The event indicator, follow-up
   time and the **Nelson-Aalen cumulative hazard** estimate are included as
   predictors, following White IR & Royston P, *Stat Med* 2009;28:1982-1998.
   Omitting the outcome biases covariate-outcome associations toward the null.
2. **The imputation model is congenial with the substantive model.** Every
   covariate in the Cox models is a predictor in the imputation model, plus
   ever-exposed indicators for each interacting drug and baseline PPI use, so
   the imputation model is at least as rich as the analysis model.
3. **Estimates are pooled by Rubin's rules** on the log-hazard scale, with
   Barnard-Rubin degrees of freedom, and the fraction of missing information
   (FMI) is reported for each coefficient.

Imputation is run **separately for each outcome cohort** (serious, minor, all
bleeding), because the outcome and follow-up time entering the imputation model
differ between them.

A **complete-case sensitivity analysis** is run alongside, as recommended by
Sterne JAC et al., *BMJ* 2009;338:b2393 and required for STROBE / RECORD-PE
reporting.


In [ ]:
# ===========================================================================
# Multiple imputation helpers (MICE + Rubin's rules)
# ===========================================================================
from scipy import stats
from lifelines import NelsonAalenFitter

# ---- Configuration --------------------------------------------------------
# m = number of imputed datasets. Set to 10 to match the manuscript and the
# White/Royston/Wood (2011) rule of thumb that m should be at least the
# percentage of incomplete cases (BMI ~8.4% missing). Set to 5 if you prefer
# the smaller number - nothing else needs to change.
MI_CONFIG = dict(
    m=10,
    iterations=5,
    seed=20240824,
    mean_match_candidates=5,   # predictive mean matching: imputed values are
                               # drawn from observed values of near neighbours
    n_jobs=10,                 # imputed datasets are independent draws, so each
                               # one is built in its own worker process. Set to 1
                               # to fall back to a single m-dataset kernel.
)
IMPUTE_COLS = ["ckd_value", "bmi_value"]   # continuous covariates to impute

# The 2 x m adjusted fits per outcome are all independent, so they are farmed
# out as a single flat pool of tasks rather than looping over imputations. Each
# worker builds its own copy of the design frame, so peak memory is roughly
# MI_FIT_N_JOBS x (size of the interval-level frame restricted to model
# columns). Lower this first if the node runs out of memory.
MI_FIT_N_JOBS = 20

# NOTE ON THREADS: the first cell of this notebook pins OMP_NUM_THREADS=1, which
# also silences LightGBM's internal threading inside miceforest. That is the
# right default here because MICE is parallelised across processes instead.
# If you set MI_CONFIG["n_jobs"]=1, give LightGBM threads back explicitly:
#     run_mice(..., num_threads=16)      # **lgb_kwargs are forwarded to mice()
# Do not do both at once - the two layers will oversubscribe the cores.


def check_mi_versions():
    """miceforest 6.0.x is incompatible with lightgbm >= 4.6 (the
    Booster.__inner_predict signature changed). Fail loudly and early rather
    than 40 minutes into a run."""
    import miceforest, lightgbm
    print(f"miceforest {miceforest.__version__} | lightgbm {lightgbm.__version__}")
    major_minor = tuple(int(x) for x in lightgbm.__version__.split(".")[:2])
    if major_minor >= (4, 6):
        print("WARNING: kernel.mice() may raise "
              "'Booster.__inner_predict() takes 1 positional argument'. "
              "If it does, run:  pip install 'lightgbm<4.6'")


# ---- 1. Nelson-Aalen cumulative hazard (White & Royston 2009) --------------
def nelson_aalen_cumhaz(durations, events):
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    naf = NelsonAalenFitter()
    naf.fit(durations, event_observed=events)
    ch = naf.cumulative_hazard_.iloc[:, 0]
    times = ch.index.to_numpy(dtype=float)
    idx = np.clip(np.searchsorted(times, durations, side="right") - 1, 0, len(times) - 1)
    return ch.to_numpy()[idx]


# ---- 2. Patient-level frame for the imputation model ----------------------
def build_imputation_frame(analysis_df, comorbidities, binary_cols, continuous_cols,
                           drug_terms=()):
    """One row per patient: total follow-up, event indicator, Nelson-Aalen
    cumulative hazard, every analysis covariate, and ever-exposed flags for the
    interacting drugs (so the imputation model is congenial with the Cox model)."""
    df = analysis_df
    terms = [t for t in list(drug_terms) if t in df.columns]

    slim = pd.DataFrame({
        "patid": df["patid"].to_numpy(),
        "_dur": (df["end_date"] - df["start_date"]).dt.days.clip(lower=0).to_numpy(),
        "event_flag": df["event_flag"].to_numpy(),
    })
    for t in terms:
        slim[t] = df[t].to_numpy().astype("int8")

    agg_map = {"fu_days": ("_dur", "sum"), "event": ("event_flag", "max")}
    agg_map.update({f"ever_{t}": (t, "max") for t in terms})
    pat = slim.groupby("patid", sort=False).agg(**agg_map).reset_index()

    pat["fu_years"] = (pat.pop("fu_days") / 365.25).clip(lower=1 / 365.25)
    pat["event"] = pat["event"].astype(int)
    pat["na_cumhaz"] = nelson_aalen_cumhaz(pat["fu_years"], pat["event"])
    pat["log_fu_years"] = np.log(pat["fu_years"])
    for t in terms:
        pat[f"ever_{t}"] = pat[f"ever_{t}"].astype(float)

    # Covariate values are taken from the ANALYSIS frame, not from the workbook.
    # That frame is what the Cox model actually sees, and it carries any
    # correction applied after the merge - notably the age backfill. Reading from
    # the workbook here would silently undo those corrections and hand MICE a
    # column it was never meant to impute. The workbook is only a fallback for
    # columns the analysis frame does not carry.
    first = analysis_df.drop_duplicates("patid").set_index("patid")
    com = (comorbidities.drop_duplicates(subset=["patid"]).set_index("patid")
           if comorbidities is not None else None)

    def _pull(col):
        if col in first.columns:
            return pd.to_numeric(pat["patid"].map(first[col]), errors="coerce")
        if com is not None and col in com.columns:
            return pd.to_numeric(pat["patid"].map(com[col]), errors="coerce")
        return None

    for col in binary_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v.fillna(0).astype(float)
    for col in continuous_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v

    return pat


# ---- 3. MICE with miceforest ----------------------------------------------
def _make_kernel(work, m, seed, mean_match_candidates=5):
    """Construct an ImputationKernel across miceforest 5.x / 6.x signatures."""
    import miceforest as mf
    attempts = [
        dict(num_datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations_data=True),
        dict(datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations=True),
        dict(num_datasets=m, random_state=seed),
        dict(datasets=m, random_state=seed),
    ]
    last = None
    for kw in attempts:
        try:
            return mf.ImputationKernel(work, **kw)
        except TypeError as exc:
            last = exc
    raise RuntimeError(f"Could not construct miceforest ImputationKernel: {last}")


_MICE_WORK = None       # published to forked workers, never pickled
_MICE_TARGETS = None


def _mice_worker(args):
    """One imputed dataset, built by its own single-dataset kernel."""
    idx, seed, iterations, mmc, lgb_kwargs = args
    kernel = _make_kernel(_MICE_WORK, 1, seed, mmc)
    kernel.mice(iterations, **lgb_kwargs)
    comp = kernel.complete_data(dataset=0)
    return idx, {c: np.asarray(comp[c], dtype=float) for c in _MICE_TARGETS}


def run_mice_parallel(pat_frame, continuous_cols, targets, m, iterations, seed,
                      mean_match_candidates, n_jobs, label="", **lgb_kwargs):
    """m independent chains in m worker processes.

    Statistically this is the same estimator as one m-dataset kernel: the m
    imputations are independent draws either way, the only difference being that
    each chain gets its own RNG stream. Streams are spawned from a single
    SeedSequence so the whole run is reproducible from MI_CONFIG["seed"]."""
    global _MICE_WORK, _MICE_TARGETS
    t0 = time.time()
    ss = np.random.SeedSequence(seed)
    seeds = [int(s.generate_state(1)[0]) % (2 ** 31 - 1) for s in ss.spawn(m)]
    tasks = [(i, seeds[i], iterations, mean_match_candidates, lgb_kwargs) for i in range(m)]

    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    _MICE_WORK, _MICE_TARGETS = work, targets
    try:
        with Pool(min(n_jobs, m)) as pool:
            results = dict(pool.map(_mice_worker, tasks))
    finally:
        _MICE_WORK = _MICE_TARGETS = None

    out = []
    for i in range(m):
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in results[i]:
                frame[c] = results[i][c]
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE: {m} datasets on {min(n_jobs, m)} workers "
          f"in {time.time() - t0:.1f}s | seeds {seeds[:3]}...")
    return out, seeds


def run_mice(pat_frame, continuous_cols, m=10, iterations=5, seed=20240824,
             mean_match_candidates=5, n_jobs=1, label="", verbose=True, **lgb_kwargs):
    """Returns (list of m patid->imputed-value frames, fitted kernel or seeds)."""
    t0 = time.time()
    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    target_cols = [c for c in work.columns if work[c].isna().any()]

    # A column with no observed values at all is a merge failure, not missing
    # data. miceforest would otherwise die inside a numpy sampling call with an
    # error that says nothing about which column is at fault.
    _empty = [c for c in target_cols if int(work[c].notna().sum()) == 0]
    if _empty:
        raise ValueError(
            f"These columns are 100% missing, so there is nothing to impute from: "
            f"{_empty}. Fix the source merge before imputing.")
    _sparse = {c: round(100 * work[c].notna().mean(), 2) for c in target_cols
               if work[c].notna().mean() < 0.5}
    if _sparse:
        print(f"[{label}] WARNING: under half observed for {_sparse} (% observed) - "
              "check the merge before trusting these imputations.")

    if verbose:
        print(f"[{label}] imputing {target_cols} | m={m}, iterations={iterations}, "
              f"predictors={work.shape[1] - 1}, n={len(work):,}")
        extra = [c for c in target_cols if c not in continuous_cols]
        if extra:
            print(f"[{label}] note: these predictors also had missing values and "
                  f"were imputed jointly: {extra}")
    if not target_cols:
        print(f"[{label}] nothing to impute - no missing values found.")
        base = pat_frame[["patid"] + [c for c in continuous_cols if c in pat_frame.columns]].copy()
        return [base.copy() for _ in range(m)], None

    if n_jobs and n_jobs > 1 and m > 1:
        return run_mice_parallel(pat_frame, continuous_cols, target_cols, m, iterations,
                                 seed, mean_match_candidates, n_jobs, label=label,
                                 **lgb_kwargs)

    kernel = _make_kernel(work, m, seed, mean_match_candidates)
    kernel.mice(iterations, **lgb_kwargs)

    out = []
    for i in range(m):
        comp = kernel.complete_data(dataset=i)
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in comp.columns:
                frame[c] = np.asarray(comp[c])
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE finished in {time.time() - t0:.1f}s")
    return out, kernel


def imputation_diagnostics(pat_frame, imputations, continuous_cols):
    """Observed vs imputed distributions - include in the supplement."""
    rows = []
    for c in continuous_cols:
        if c not in pat_frame.columns:
            continue
        miss = pat_frame[c].isna().to_numpy()
        obs = pat_frame.loc[~miss, c]
        rows.append(dict(variable=c, source="observed", n=int(obs.size), mean=obs.mean(),
                         sd=obs.std(), p25=obs.quantile(.25), median=obs.median(),
                         p75=obs.quantile(.75)))
        if miss.sum() == 0:
            continue
        for i, imp in enumerate(imputations, 1):
            v = pd.to_numeric(imp.loc[miss, c])
            rows.append(dict(variable=c, source=f"imputed {i}", n=int(v.size), mean=v.mean(),
                             sd=v.std(), p25=v.quantile(.25), median=v.median(),
                             p75=v.quantile(.75)))
    return pd.DataFrame(rows).round(3)


# ---- 4. Rubin's rules ------------------------------------------------------
def rubin_pool(summaries, n_events=None, n_params=None):
    """Pool lifelines `.summary` frames on the log-hazard scale.

    Returns a frame shaped like a lifelines summary (so it can be passed
    straight to build_bleeding_summary) plus FMI, RIV and Barnard-Rubin df."""
    coef = pd.concat([s["coef"] for s in summaries], axis=1)
    se = pd.concat([s["se(coef)"] for s in summaries], axis=1)
    m = coef.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")

    qbar = coef.mean(axis=1)                 # pooled point estimate
    ubar = (se ** 2).mean(axis=1)            # within-imputation variance
    b = coef.var(axis=1, ddof=1)             # between-imputation variance
    total = ubar + (1 + 1 / m) * b           # total variance
    se_pool = np.sqrt(total)

    lam = np.where(total > 0, ((1 + 1 / m) * b) / total, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(ubar > 0, ((1 + 1 / m) * b) / ubar, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)

    if n_events and n_params:                # Barnard & Rubin (1999) adjustment
        nu_com = max(int(n_events) - int(n_params), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    out = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "coef lower 95%": qbar - crit * se_pool,
        "coef upper 95%": qbar + crit * se_pool,
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "riv": r,
        "df": df_adj,
        "n_imputations": m,
    })
    out.index.name = "covariate"
    return out


# ---- 5. Drivers: fit across imputations, pool, and complete-case fallback --
_MI_BASE_DF = None      # inherited copy-on-write by forked workers
_MI_VALUES = None       # per-imputation, row-aligned arrays of imputed values


def _fit_mi_worker(args):
    imp_idx, key, formula, cols = args
    sub = _MI_BASE_DF[cols].copy()
    for c, arr in _MI_VALUES[imp_idx].items():
        if c in sub.columns:
            sub[c] = arr
    ctv = CoxTimeVaryingFitter()
    ctv.fit(sub, id_col="patid", start_col="start_date", stop_col="end_date",
            event_col="event_flag", formula=formula, show_progress=False)
    return imp_idx, key, ctv.summary


def fit_adjusted_models_mi(df, imputations, impute_cols=None, specs=None,
                           n_jobs=None, label=""):
    """Refit both adjusted models on every imputed dataset and pool.

    The 2 x m fits are independent, so they run as one flat pool of tasks. The
    imputed values are mapped onto the interval-level rows ONCE in the parent
    and inherited copy-on-write; each worker only pays for its own design frame.
    `df` itself is never modified."""
    impute_cols = IMPUTE_COLS if impute_cols is None else list(impute_cols)
    specs = MODEL_SPECS if specs is None else specs
    n_jobs = MI_FIT_N_JOBS if n_jobs is None else n_jobs
    global _MI_BASE_DF, _MI_VALUES
    t0 = time.time()

    _model_cols = sorted({c for _, cols in specs.values() for c in cols}
                         - {"patid", "start_date", "end_date", "event_flag"})

    values = []
    for i, imp in enumerate(imputations, 1):
        lut = imp.set_index("patid")
        vals = {c: df["patid"].map(lut[c]).to_numpy(dtype=float)
                for c in impute_cols if c in lut.columns}
        for c, arr in vals.items():
            n_bad = int(np.isnan(arr).sum())
            if n_bad:
                raise ValueError(f"{c}: {n_bad} rows still NaN after imputation {i} - "
                                 "some patids are absent from the imputation frame.")
        values.append(vals)

    _still = {c: int(df[c].isna().sum()) for c in _model_cols
              if c not in impute_cols and df[c].isna().any()}
    if _still:
        raise ValueError(f"Model covariates outside IMPUTE_COLS still contain NaN: {_still}. "
                         "Add them to IMPUTE_COLS so they are written back from the "
                         "imputed datasets.")

    tasks = [(i, k, f, c) for i in range(len(imputations)) for k, (f, c) in specs.items()]
    _MI_BASE_DF, _MI_VALUES = df, values
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            res = pool.map(_fit_mi_worker, tasks)
    finally:
        _MI_BASE_DF = _MI_VALUES = None

    per_model = {k: [None] * len(imputations) for k in specs}
    for imp_idx, key, summ in res:
        per_model[key][imp_idx] = summ

    n_events = int(df.loc[df["event_flag"] == 1, "patid"].nunique())
    pooled = {k: rubin_pool(v, n_events=n_events, n_params=len(v[0]))
              for k, v in per_model.items()}
    print(f"[{label}] {len(tasks)} fits on {min(n_jobs, len(tasks))} workers, "
          f"pooled in {time.time() - t0:.1f}s")
    return pooled, per_model


def complete_case_frame(df, impute_cols=None, label=""):
    impute_cols = IMPUTE_COLS if impute_cols is None else impute_cols
    mask = df[list(impute_cols)].notna().all(axis=1)
    sub = df.loc[mask].reset_index(drop=True)
    print(f"[{label}] complete-case: {sub['patid'].nunique():,} of "
          f"{df['patid'].nunique():,} patients "
          f"({100 * sub['patid'].nunique() / df['patid'].nunique():.1f}%), "
          f"{int(sub.loc[sub['event_flag'] == 1, 'patid'].nunique()):,} events retained")
    return sub


def compare_mi_vs_cc(mi_summary, cc_summary):
    """Side-by-side MI vs complete-case HRs with the percentage change in the
    point estimate - this is the table reviewers ask for."""
    a = mi_summary[["drug_class", "HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (MI)", "CI Lower": "CI Lower (MI)",
                 "CI Upper": "CI Upper (MI)", "p": "p (MI)"})
    b = cc_summary[["drug_class", "HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (CC)", "CI Lower": "CI Lower (CC)",
                 "CI Upper": "CI Upper (CC)", "p": "p (CC)"})
    out = a.merge(b, on="drug_class", how="outer")
    out["% change in HR"] = (100 * (out["HR (CC)"] - out["HR (MI)"]) / out["HR (MI)"]).round(1)
    return out.sort_values("HR (MI)", ascending=False).reset_index(drop=True)


check_mi_versions()


In [ ]:
# ---- Adapter: this notebook fits ONE adjusted model per outcome -------------
# The shared MI helpers above are written against a dict of model specs, because
# . Here there is a single adjusted model, so the spec
# dict has one entry and everything downstream works unchanged.
MODEL_SPECS = {"adjusted": (covariates, adjusted_cols)}


def attach_mi_diagnostics_single(summary, pooled, key="adjusted"):
    """Add FMI / Barnard-Rubin df / m to a summary table."""
    p = pooled[key]
    recs = [{"drug_class": d,
             "FMI": round(float(p.loc[d, "fmi"]), 3),
             "MI df": round(float(p.loc[d, "df"]), 1),
             "m": int(p.loc[d, "n_imputations"])}
            for d in summary["drug_class"]]
    return summary.merge(pd.DataFrame(recs), on="drug_class", how="left")


def crude_hr_table(df, crude_terms, n_jobs=None, label=""):
    """Crude HRs contain no imputed covariate, so they are fitted once on the
    observed data rather than once per imputation. Reuses this notebook's own
    _fit_worker / _FIT_DF fork pattern."""
    global _FIT_DF
    crude_terms = list(crude_terms)
    tasks = [("crude", t, t, _INDEX_COLS + [t]) for t in crude_terms]
    t0 = time.time()
    _FIT_DF = df
    try:
        with Pool(min(n_jobs or len(tasks), len(tasks))) as pool:
            results = pool.map(_fit_worker, tasks)
    finally:
        _FIT_DF = None
    rows = {n: r for _, n, r in results}
    print(f"[{label}] {len(tasks)} crude models in {time.time() - t0:.1f}s")
    return pd.DataFrame([rows[t] for t in crude_terms])


### Minor Bleeding


In [ ]:
minor_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx"), dtype=str)
minor_doac_exposure["age_at_study_start"] = minor_doac_exposure["age_at_study_start"].astype(int)
minor_doac_exposure["study_start_date"] = pd.to_datetime(minor_doac_exposure["study_start_date"])
minor_doac_exposure["study_end_date"] = pd.to_datetime(minor_doac_exposure["study_end_date"])
minor_doac_exposure["event_date"] = pd.to_datetime(minor_doac_exposure["event_date"])

minor_doac_exposure = minor_doac_exposure[(minor_doac_exposure["age_at_study_start"] >= 18) &
                (minor_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (minor_doac_exposure["taken_warfarin_1w_before"] =="0") &
                    (minor_doac_exposure["taken_warfarin_during_study"] =="0")]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
minor_doac_exposure = minor_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]

In [ ]:
# ===========================================================================
# COMPETING EVENT: censor minor-bleeding follow-up at the first SERIOUS bleed
# ===========================================================================
# Reviewer comment 2. A first MINOR bleed cannot be observed once a serious
# bleed has occurred, and post-serious-bleed follow-up is a different population
# (anticoagulation often interrupted, monitoring intensified). So for THIS
# outcome only, serious bleeding is a competing event and follow-up stops there.
# Death is already handled through study_end_date in the cohort workbook.
#
# Applies to the minor-bleeding cohort ONLY. The serious and all-bleeding
# analyses are unchanged: death is their only competing event and it is already
# a censoring event, so those models already estimate cause-specific hazards.
#
# This cell must run BEFORE the minor prescriptions are re-clipped and merged,
# because those cells map study_end_date from minor_doac_exposure.

_major_dates = (major_events_df.drop_duplicates(subset=["patid"])
                               .set_index("patid")["event_date"])
minor_doac_exposure = minor_doac_exposure.copy()
minor_doac_exposure["major_event_date"] = pd.to_datetime(
    minor_doac_exposure["patid"].map(_major_dates), errors="coerce")

_start = minor_doac_exposure["study_start_date"]
_end   = minor_doac_exposure["study_end_date"]
_comp  = minor_doac_exposure["major_event_date"]
_event = minor_doac_exposure["event_date"]

# The competing event applies whenever the serious bleed falls inside this
# patient's follow-up window, WHETHER OR NOT a minor bleed was ever recorded.
# Testing `major_event_date <= event_date` instead would be False for every
# patient with event_date = NaT, leaving the largest affected group - those with
# a serious bleed and no later minor bleed - contributing person-time after
# their serious bleed.
_in_fu = _comp.notna() & (_comp >= _start) & (_comp <= _end)

_new_end = _end.where(~_in_fu, _comp)

# Same-day policy: a minor bleed coded on the day of the serious bleed is
# usually the same clinical episode, so it is retained (event_date <= new end).
# Use `< _new_end` instead to discard same-day minor bleeds.
_new_event = _event.where(_event.notna() & (_event <= _new_end), pd.NaT)

_rep = {
    "patients": int(len(minor_doac_exposure)),
    "serious bleed during minor follow-up": int(_in_fu.sum()),
    "follow-up truncated": int((_new_end < _end).sum()),
    "person-years removed": round(float(((_end - _new_end).dt.total_seconds() / 86400)
                                        .clip(lower=0).sum()) / 365.25, 1),
    "minor events before": int(_event.notna().sum()),
    "minor events after": int(_new_event.notna().sum()),
    "minor events now censored": int(_event.notna().sum() - _new_event.notna().sum()),
}

minor_doac_exposure["study_end_date"] = _new_end
minor_doac_exposure["event_date"] = _new_event

# A serious bleed on the index date leaves no follow-up at all.
_zero = minor_doac_exposure["study_end_date"] <= minor_doac_exposure["study_start_date"]
_rep["patients dropped (zero follow-up)"] = int(_zero.sum())
minor_doac_exposure = minor_doac_exposure.loc[~_zero].reset_index(drop=True)

competing_event_report = pd.Series(_rep).to_frame("n")
display(competing_event_report)

In [ ]:
# ===========================================================================
# Checks on the competing-event censoring - run before trusting anything below
# ===========================================================================
_s = minor_doac_exposure

_past = int((_s["major_event_date"].notna() &
             (_s["study_end_date"] > _s["major_event_date"])).sum())
print(f"1. patients still followed past their serious bleed : {_past}  "
      f"{'OK' if _past == 0 else '<-- BUG, do not proceed'}")

_after = int((_s["event_date"].notna() & (_s["event_date"] > _s["study_end_date"])).sum())
print(f"2. retained minor events after end of follow-up     : {_after}  "
      f"{'OK' if _after == 0 else '<-- BUG'}")

# Does follow-up stop at the minor bleed itself? If study_end_date is later than
# event_date for cases, person-time is being counted after the outcome - a
# separate problem from competing events, but worth knowing about.
_cases = _s[_s["event_date"].notna()]
_beyond = int((_cases["study_end_date"] > _cases["event_date"]).sum())
print(f"3. cases whose follow-up continues past their event : {_beyond}  "
      f"{'OK' if _beyond == 0 else '<-- investigate'}")

# Minor follow-up must now be <= major follow-up for everyone. This also fixes a
# pre-existing problem: interacting_drugs_exposure was built with the MAJOR
# cohort's study_end_date, so prescriptions issued after that date were never
# loaded. Where minor follow-up used to run past it, that exposure was missing
# and the person-time counted as drug-free.
_mj = major_doac_exposure.drop_duplicates("patid").set_index("patid")["study_end_date"]
_cmp = _s.assign(major_end=_s["patid"].map(_mj))
_longer = int((_cmp["study_end_date"] > _cmp["major_end"]).sum())
print(f"4. patients whose minor follow-up exceeds major     : {_longer}  "
      f"{'OK' if _longer == 0 else '<-- exposure may be missing for these patients'}")

_n = len(_s)
_ev = int(_s["event_date"].notna().sum())
_cens_comp = int((_s["event_date"].isna() & _s["major_event_date"].notna() &
                  (_s["major_event_date"] <= _s["study_end_date"])).sum())
print(f"\n5. minor-bleeding cohort: {_n:,} patients | {_ev:,} minor bleeds | "
      f"{_cens_comp:,} censored at a serious bleed | "
      f"{_n - _ev - _cens_comp:,} censored otherwise (death, discontinuation, end of study)")

In [ ]:
minor_interacting_drugs_presc = interacting_drugs_exposure.copy()
minor_interacting_drugs_presc["study_end_date"] = minor_interacting_drugs_presc["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
minor_interacting_drugs_presc = minor_interacting_drugs_presc[minor_interacting_drugs_presc["issuedate"] < minor_interacting_drugs_presc["study_end_date"]]
minor_interacting_drugs_presc.sort_values(['patid', 'issuedate'], inplace=True)
minor_interacting_drugs_presc.reset_index(drop=True, inplace=True)
minor_interacting_drugs_presc

In [ ]:
minor_interacting_drugs_presc["pk_drug"] = minor_interacting_drugs_presc["drug_name"].isin(p_k["Drug"]).astype(int)
minor_interacting_drugs_presc["pd_drug"] = minor_interacting_drugs_presc["drug_name"].isin(p_d["Drug"]).astype(int)
minor_interacting_drugs_presc["pk_pd"] = np.where(minor_interacting_drugs_presc["pk_drug"]==1, "pk", "pd")
minor_interacting_drugs_presc.drop(columns=["pk_drug", "pd_drug"], inplace=True)


minor_interacting_drugs_presc_merged = merge_prescriptions(minor_interacting_drugs_presc)
minor_interacting_drugs_presc_merged["study_start_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_interacting_drugs_presc_merged["study_end_date"] = minor_interacting_drugs_presc_merged["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

minor_interacting_drugs_presc_merged

In [ ]:
minor_exposure_timeline_df = build_exposure_timeline(minor_interacting_drugs_presc_merged)
minor_exposure_timeline_df["study_start_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_start_date"])
minor_exposure_timeline_df["study_end_date"] = minor_exposure_timeline_df["patid"].map(minor_doac_exposure.set_index("patid")["study_end_date"])

In [ ]:
minor_events_df = minor_doac_exposure[minor_doac_exposure["event_date"].notnull()]

minor_exposure_timeline_df = minor_exposure_timeline_df.merge(minor_events_df[["patid", "event_date"]], on="patid", how="left")

minor_exposure_timeline_df["event_flag"] = np.where(
    (minor_exposure_timeline_df["event_date"] >= minor_exposure_timeline_df["start_date"]) & 
    (minor_exposure_timeline_df["event_date"] <= minor_exposure_timeline_df["end_date"]), 1, 0)

minor_exposure_timeline_df["p_k_drug"] = minor_exposure_timeline_df["concomitant_drugs_ls"].apply(
    lambda drugs: any(drug in p_k["Drug"].values for drug in drugs)
)
minor_exposure_timeline_df["p_d_drug"] = minor_exposure_timeline_df["concomitant_drugs_ls"].apply(
    lambda drugs: any(drug in p_d["Drug"].values for drug in drugs)
)

minor_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:
minor_exposure_timeline_df = add_never_exposed(
    minor_exposure_timeline_df, minor_doac_exposure,
    exposure_zero_cols=("p_k_drug", "p_d_drug"),
    label="Minor bleeding")


In [ ]:
minor_df = minor_exposure_timeline_df.copy()

for col in binary_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

for col in continuous_comorbidity_cols:
    minor_df[col] = minor_df["patid"].map(comorbidities.set_index("patid")[col])
minor_df = backfill_age(minor_df, minor_doac_exposure, "Minor bleeding")
missing_report_minor = report_missingness(minor_df, "Minor bleeding")


In [ ]:
# ---- MICE for the Minor bleeding cohort ---------------------------------
# Imputation is cohort-specific: the outcome and follow-up time in the
# imputation model must match the model the imputed values will be used in.
# The exposure terms enter as ever-exposed flags so that the imputation model is
# congenial with the Cox model.
minor_imp_frame = build_imputation_frame(
    minor_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_classes_to_consider)

print(minor_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

minor_imputations, minor_mice_seeds = run_mice(
    minor_imp_frame, IMPUTE_COLS, label="Minor bleeding", **MI_CONFIG)

minor_imp_diag = imputation_diagnostics(minor_imp_frame, minor_imputations, IMPUTE_COLS)
minor_imp_diag


### Minor bleeding - imputed and pooled estimates


In [ ]:
# ---- Adjusted model across imputations, pooled by Rubin's rules ----------
minor_pooled_fits, minor_per_imp_fits = fit_adjusted_models_mi(
    minor_df, minor_imputations, label="Minor bleeding")

crude_hr_minor_df = crude_hr_table(minor_df, drug_classes_to_consider, label="Minor bleeding")

minor_summary = build_bleeding_summary(minor_pooled_fits["adjusted"], minor_df,
                                   drug_classes_to_consider, all_terms=drug_classes_to_consider)
minor_summary = attach_mi_diagnostics_single(minor_summary, minor_pooled_fits)
minor_summary = minor_summary.merge(crude_hr_minor_df, on="drug_class", how="left")
minor_summary.to_excel(os.path.join(SENS_COMPETING_DIR, "hr_drug_pk_pd_minor_summary.xlsx"), index=False)
minor_summary
